In [2]:
!pip install inflection

In [3]:
# bibliotecas necessárias
import pandas as pd
import inflection
import io
import re
import plotly.express  as px
import folium
from google import colab as cl

# importando o arquivo para o computador
file_upload = cl.files.upload()

Saving zomato.csv to zomato.csv


In [4]:
df = pd.read_csv('zomato.csv')
df1 = df.copy()

In [5]:
# Para colocar o nome dos países com base no código de cada país

COUNTRIES = {
1: "India",
14: "Australia",
30: "Brazil",
37: "Canada",
94: "Indonesia",
148: "New Zeland",
162: "Philippines",
166: "Qatar",
184: "Singapure",
189: "South Africa",
191: "Sri Lanka",
208: "Turkey",
214: "United Arab Emirates",
215: "England",
216: "United States of America",
}
def country_name(country_id):
  return COUNTRIES[country_id]

df1['Country Code'] = df1.loc[:, 'Country Code'].apply(lambda x: country_name(x))

In [6]:
# Criar a categoria do tipo de comida com base no range de valores
def create_price_tye(price_range):
  if price_range == 1:
    return "cheap"
  elif price_range == 2:
    return "normal"
  elif price_range == 3:
    return "expensive"
  else:
    return "gourmet"

df1['Price range'] = df1.loc[:, 'Price range'].apply(lambda x: create_price_tye(x))

In [7]:
# Criar o nome das cores com base nos códigos de cores
COLORS = {
"3F7E00": "darkgreen",
"5BA829": "green",
"9ACD32": "lightgreen",
"CDD614": "orange",
"FFBA00": "red",
"CBCBC8": "darkred",
"FF7800": "darkred",
}
def color_name(color_code):
  return COLORS[color_code]

df1['Expressed color'] = df1.loc[:, 'Rating color'].apply(lambda x: color_name(x))

In [8]:
# Renomear as colunas do DataFrame
def rename_columns(dataframe):
    df = dataframe.copy()
    title = lambda x: inflection.titleize(x)
    snakecase = lambda x: inflection.underscore(x)
    spaces = lambda x: x.replace(" ", "")
    cols_old = list(df.columns)
    cols_old = list(map(title, cols_old))
    cols_old = list(map(spaces, cols_old))
    cols_new = list(map(snakecase, cols_old))
    df.columns = cols_new
    return df

df1 = rename_columns(df1)

In [9]:
# Verificando dados faltantes por coluna
missing_data_per_column = df1.isnull().sum()
print("Dados faltantes por coluna:")
print(missing_data_per_column)

# Verificando o total de dados faltantes no DataFrame
total_missing_data = df1.isnull().sum().sum()
print(f"\nTotal de dados faltantes no DataFrame: {total_missing_data}")

Dados faltantes por coluna:
restaurant_id            0
restaurant_name          0
country_code             0
city                     0
address                  0
locality                 0
locality_verbose         0
longitude                0
latitude                 0
cuisines                15
average_cost_for_two     0
currency                 0
has_table_booking        0
has_online_delivery      0
is_delivering_now        0
switch_to_order_menu     0
price_range              0
aggregate_rating         0
rating_color             0
rating_text              0
votes                    0
expressed_color          0
dtype: int64

Total de dados faltantes no DataFrame: 15


In [10]:
# Removendo a coluna 'Switch to order menu' pois não possui valores
df1 = df1.drop(labels=['switch_to_order_menu'], axis='columns')

In [11]:
#Categorizar todos os restaurantes somente por um tipo de culinária
df1['cuisines'] = df1['cuisines'].astype(str)
df1["cuisines"] = df1.loc[:, "cuisines"].apply(lambda x: x.split(",")[0])

In [12]:
# Removendo os NaN da coluna Cuisines
df1 = df1.loc[(df1['cuisines'] != 'nan'),:].copy()
df1 = df1.loc[(df1['cuisines'] != 'Drinks Only'),:].copy()
df1 = df1.loc[(df1['cuisines'] != 'Mineira'),:].copy()

In [13]:
# Removendo linhas duplicadas do DataFrame
df1 = df1.drop_duplicates().reset_index(drop=True)

In [14]:
# Verificando dados faltantes por coluna
missing_data_per_column = df1.isnull().sum()
print("Dados faltantes por coluna:")
print(missing_data_per_column)

# Verificando o total de dados faltantes no DataFrame
total_missing_data = df1.isnull().sum().sum()
print(f"\nTotal de dados faltantes no DataFrame: {total_missing_data}")

Dados faltantes por coluna:
restaurant_id           0
restaurant_name         0
country_code            0
city                    0
address                 0
locality                0
locality_verbose        0
longitude               0
latitude                0
cuisines                0
average_cost_for_two    0
currency                0
has_table_booking       0
has_online_delivery     0
is_delivering_now       0
price_range             0
aggregate_rating        0
rating_color            0
rating_text             0
votes                   0
expressed_color         0
dtype: int64

Total de dados faltantes no DataFrame: 0


# GERAL

In [ ]:
colunas = list(df1.columns)
for col in colunas:
    print(f'{col} - {len(df1[col].unique())}')

restaurant_id - 6927
restaurant_name - 5899
country_code - 15
city - 125
address - 6751
locality - 2272
locality_verbose - 2356
longitude - 6831
latitude - 6818
cuisines - 163
average_cost_for_two - 156
currency - 12
has_table_booking - 2
has_online_delivery - 2
is_delivering_now - 2
price_range - 4
aggregate_rating - 30
rating_color - 7
rating_text - 28
votes - 1739
expressed_color - 6


In [ ]:
#1. Quantos restaurantes estão registrados?
unique_restaurants = df1['restaurant_id'].nunique()
print(f"O número de restaurantes únicos é: {unique_restaurants}")



O número de restaurantes únicos é: 6927


In [ ]:
#2. Quantos países únicos estão registrados?
unique_countries = df1['country_code'].nunique()
print(f"O número de países únicos é: {unique_countries}")

O número de países únicos é: 15


In [ ]:
#3. Quantas cidades únicas estão registradas?
unique_cities = df1['city'].nunique()
print(f"O número de cidades únicas é: {unique_cities}")


O número de cidades únicas é: 125


In [ ]:
#4. Qual o total de avaliações feitas?
total_ratings = df1['votes'].sum()
print(f"O total de avaliações feitas é: {total_ratings}")

O total de avaliações feitas é: 4194530


In [ ]:
#5.Mostre quantos tipos de culinária únicos?
unique_cuisines = df1['cuisines'].nunique()
print(f"O total de culinarias unicas é: {unique_cuisines}")

O total de culinarias unicas é: 163


# PAIS

In [ ]:
#1. Qual o nome do país que possui mais cidades registradas?
most_cities_country_row = country_city_counts.loc[country_city_counts['city'].idxmax()]
most_cities_country_name = most_cities_country_row['country_code']
most_cities_count = most_cities_country_row['city']

print(f"O país com a maior quantidade de cidades registradas é '{most_cities_country_name}' com {most_cities_count} cidades.")

O país com a maior quantidade de cidades registradas é 'India' com 49 cidades.


In [ ]:
# 2. Qual o nome do país que possui mais restaurantes registrados?
restaurants_per_country = df1.groupby('country_code')['restaurant_id'].nunique().reset_index()

most_restaurants_country_row = restaurants_per_country.loc[restaurants_per_country['restaurant_id'].idxmax()]
most_restaurants_country_name = most_restaurants_country_row['country_code']
most_restaurants_count = most_restaurants_country_row['restaurant_id']

print(f"O país com a maior quantidade de restaurantes registrados é '{most_restaurants_country_name}' com {most_restaurants_count} restaurantes.")

O país com a maior quantidade de restaurantes registrados é 'India' com 3111 restaurantes.


In [ ]:
# 3. Qual o nome do país que possui mais restaurantes com o nível de preço igual a 4 registrados?
restaurants_price_4 = df1[df1['price_range'] == 'gourmet'] # 'gourmet' corresponds to price_range 4

restaurants_price_4_per_country = restaurants_price_4.groupby('country_code')['restaurant_id'].nunique().reset_index()

most_restaurants_price_4_country_row = restaurants_price_4_per_country.loc[restaurants_price_4_per_country['restaurant_id'].idxmax()]
most_restaurants_price_4_country_name = most_restaurants_price_4_country_row['country_code']
most_restaurants_price_4_count = most_restaurants_price_4_country_row['restaurant_id']

print(f"O país com a maior quantidade de restaurantes com nível de preço 'gourmet' (4) é '{most_restaurants_price_4_country_name}' com {most_restaurants_price_4_count} restaurantes.")

O país com a maior quantidade de restaurantes com nível de preço 'gourmet' (4) é 'United States of America' com 415 restaurantes.


In [ ]:
# 4. Qual o nome do país que possui a maior quantidade de tipos de culinária distintos?
cuisines_per_country = df1.groupby('country_code')['cuisines'].nunique().reset_index()

most_diverse_country_row = cuisines_per_country.loc[cuisines_per_country['cuisines'].idxmax()]
most_diverse_country_name = most_diverse_country_row['country_code']
most_diverse_cuisines_count = most_diverse_country_row['cuisines']

print(f"O país com a maior quantidade de tipos de culinária distintos é '{most_diverse_country_name}' com {most_diverse_cuisines_count} culinárias.")

O país com a maior quantidade de tipos de culinária distintos é 'India' com 77 culinárias.


In [ ]:
# 5. Qual o nome do país que possui a maior quantidade de avaliações feitas?
votes_per_country = df1.groupby('country_code')['votes'].sum().reset_index()

most_voted_country_row = votes_per_country.loc[votes_per_country['votes'].idxmax()]
most_voted_country_name = most_voted_country_row['country_code']
most_voted_count = most_voted_country_row['votes']

print(f"O país com a maior quantidade de avaliações feitas é '{most_voted_country_name}' com um total de {most_voted_count} avaliações.")

O país com a maior quantidade de avaliações feitas é 'India' com um total de 2800164 avaliações.


In [ ]:
# 6. Qual o nome do país que possui a maior quantidade de restaurantes que fazem entrega?

# Filtrando restaurantes que fazem entrega (assumindo 1 para 'Sim' ou True)
restaurants_with_delivery = df1[df1['has_online_delivery'] == 1]

# Agrupando por país e contando restaurantes únicos que fazem entrega
delivery_restaurants_per_country = restaurants_with_delivery.groupby('country_code')['restaurant_id'].nunique().reset_index()

# Encontrando o país com a maior quantidade
most_delivery_country_row = delivery_restaurants_per_country.loc[delivery_restaurants_per_country['restaurant_id'].idxmax()]
most_delivery_country_name = most_delivery_country_row['country_code']
most_delivery_count = most_delivery_country_row['restaurant_id']

print(f"O país com a maior quantidade de restaurantes que fazem entrega é '{most_delivery_country_name}' com {most_delivery_count} restaurantes.")

O país com a maior quantidade de restaurantes que fazem entrega é 'India' com 2177 restaurantes.


In [ ]:
# 7. Qual o nome do país que possui a maior quantidade de restaurantes que aceitam reservas?

# Filtrando restaurantes que aceitam reservas (assumindo 1 para 'Sim' ou True)
restaurants_with_booking = df1[df1['has_table_booking'] == 1]

# Agrupando por país e contando restaurantes únicos que aceitam reservas
booking_restaurants_per_country = restaurants_with_booking.groupby('country_code')['restaurant_id'].nunique().reset_index()

# Encontrando o país com a maior quantidade
most_booking_country_row = booking_restaurants_per_country.loc[booking_restaurants_per_country['restaurant_id'].idxmax()]
most_booking_country_name = most_booking_country_row['country_code']
most_booking_count = most_booking_country_row['restaurant_id']

print(f"O país com a maior quantidade de restaurantes que aceitam reservas é '{most_booking_country_name}' com {most_booking_count} restaurantes.")

O país com a maior quantidade de restaurantes que aceitam reservas é 'India' com 256 restaurantes.


In [ ]:
# 8. Qual o nome do país que possui, na média, a maior quantidade de avaliações registrada?

# Agrupando por país e calculando a média de votos
average_votes_per_country = df1.groupby('country_code')['votes'].mean().reset_index()

# Encontrando o país com a maior média de votos
most_average_voted_country_row = average_votes_per_country.loc[average_votes_per_country['votes'].idxmax()]
most_average_voted_country_name = most_average_voted_country_row['country_code']
most_average_voted_count = most_average_voted_country_row['votes']

print(f"O país com a maior média de avaliações por restaurante é '{most_average_voted_country_name}' com uma média de {most_average_voted_count:.2f} avaliações.")

O país com a maior média de avaliações por restaurante é 'Indonesia' com uma média de 1112.83 avaliações.


In [ ]:
# 9. Qual o nome do país que possui, na média, a maior nota média registrada?

# Agrupando por país e calculando a média da avaliação agregada
average_rating_per_country = df1.groupby('country_code')['aggregate_rating'].mean().reset_index()

# Encontrando o país com a maior média de avaliação
most_average_rated_country_row = average_rating_per_country.loc[average_rating_per_country['aggregate_rating'].idxmax()]
most_average_rated_country_name = most_average_rated_country_row['country_code']
most_average_rated_count = most_average_rated_country_row['aggregate_rating']

print(f"O país com a maior nota média de avaliação por restaurante é '{most_average_rated_country_name}' com uma média de {most_average_rated_count:.2f}.")

O país com a maior nota média de avaliação por restaurante é 'Indonesia' com uma média de 4.60.


In [ ]:
# 10. Qual o nome do país que possui, na média, a menor nota média registrada?

# Agrupando por país e calculando a média da avaliação agregada
average_rating_per_country = df1.groupby('country_code')['aggregate_rating'].mean().reset_index()

# Encontrando o país com a menor média de avaliação
least_average_rated_country_row = average_rating_per_country.loc[average_rating_per_country['aggregate_rating'].idxmin()]
least_average_rated_country_name = least_average_rated_country_row['country_code']
least_average_rated_count = least_average_rated_country_row['aggregate_rating']

print(f"O país com a menor nota média de avaliação por restaurante é '{least_average_rated_country_name}' com uma média de {least_average_rated_count:.2f}.")

O país com a menor nota média de avaliação por restaurante é 'Brazil' com uma média de 3.34.


In [ ]:
# 11. Qual a média de preço de um prato para dois por país?

# Agrupando por país e calculando a média do custo para dois
average_cost_for_two_per_country = df1.groupby('country_code')['average_cost_for_two'].mean().reset_index()

# Ordenando os resultados por ordem decrescente e exibindo
print("Média de preço de um prato para dois por país (ordenado decrescentemente):")
display(average_cost_for_two_per_country.sort_values(by='average_cost_for_two', ascending=False).round(2))

Média de preço de um prato para dois por país (ordenado decrescentemente):


,country_code,average_cost_for_two
5,Indonesia,303000.00
0,Australia,138959.78
11,Sri Lanka,2579.38
7,Philippines,1227.82
4,India,704.40
10,South Africa,339.52
8,Qatar,174.00
13,United Arab Emirates,153.72
9,Singapure,141.44
1,Brazil,139.39


# CIDADE

In [ ]:
#1. Qual o nome da cidade que possui mais restaurantes registrados?
restaurants_per_city = df1.groupby('city')['restaurant_id'].nunique().reset_index()

most_restaurants_city_row = restaurants_per_city.loc[restaurants_per_city['restaurant_id'].idxmax()]
most_restaurants_city_name = most_restaurants_city_row['city']
most_restaurants_count = most_restaurants_city_row['restaurant_id']

print(f"A cidade com a maior quantidade de restaurantes registrados é '{most_restaurants_city_name}' com {most_restaurants_count} restaurantes.")

A cidade com a maior quantidade de restaurantes registrados é 'Abu Dhabi' com 80 restaurantes.


In [ ]:
# 2. Qual o nome da cidade que possui mais restaurantes com nota média acima de 4?
# Agrupando por cidade e contando restaurantes únicos com avaliação acima de 4
restaurants_above_4_per_city = restaurants_above_4_rating.groupby('city')['restaurant_id'].nunique().reset_index()

# Encontrando a cidade com a maior quantidade
most_rated_restaurants_city_row = restaurants_above_4_per_city.loc[restaurants_above_4_per_city['restaurant_id'].idxmax()]
most_rated_restaurants_city_name = most_rated_restaurants_city_row['city']
most_rated_restaurants_count = most_rated_restaurants_city_row['restaurant_id']

print(f"A cidade com a maior quantidade de restaurantes com nota média acima de 4 é '{most_rated_restaurants_city_name}' com {most_rated_restaurants_count} restaurantes.")

A cidade com a maior quantidade de restaurantes com nota média acima de 4 é 'Bangalore' com 79 restaurantes.


In [ ]:
# 3. Qual o nome da cidade que possui mais restaurantes com nota média abaixo de 2.5?

# Filtrando restaurantes com nota média abaixo de 2.5
restaurants_below_2_5_rating = df1[df1['aggregate_rating'] < 2.5]

# Agrupando por cidade e contando restaurantes únicos com avaliação abaixo de 2.5
restaurants_below_2_5_per_city = restaurants_below_2_5_rating.groupby('city')['restaurant_id'].nunique().reset_index()

# Encontrando a cidade com a maior quantidade
if not restaurants_below_2_5_per_city.empty:
    most_low_rated_restaurants_city_row = restaurants_below_2_5_per_city.loc[restaurants_below_2_5_per_city['restaurant_id'].idxmax()]
    most_low_rated_restaurants_city_name = most_low_rated_restaurants_city_row['city']
    most_low_rated_restaurants_count = most_low_rated_restaurants_city_row['restaurant_id']
    print(f"A cidade com a maior quantidade de restaurantes com nota média abaixo de 2.5 é '{most_low_rated_restaurants_city_name}' com {most_low_rated_restaurants_count} restaurantes.")
else:
    print("Não há cidades com restaurantes com nota média abaixo de 2.5.")

A cidade com a maior quantidade de restaurantes com nota média abaixo de 2.5 é 'Gangtok' com 33 restaurantes.


In [ ]:
# 4. Qual o nome da cidade que possui o maior valor médio de um prato para dois?

# Agrupando por cidade e calculando a média do custo para dois
average_cost_for_two_per_city = df1.groupby('city')['average_cost_for_two'].mean().reset_index()

# Encontrando a cidade com o maior valor médio
most_expensive_city_row = average_cost_for_two_per_city.loc[average_cost_for_two_per_city['average_cost_for_two'].idxmax()]
most_expensive_city_name = most_expensive_city_row['city']
most_expensive_city_average_cost = most_expensive_city_row['average_cost_for_two']

print(f"A cidade com o maior valor médio de um prato para dois é '{most_expensive_city_name}' com um custo médio de {most_expensive_city_average_cost:.2f}.")

A cidade com o maior valor médio de um prato para dois é 'Adelaide' com um custo médio de 416734.13.


In [ ]:
# 5. Qual o nome da cidade que possui a maior quantidade de tipos de culinária distintas?

# Agrupando por cidade e contando o número de culinárias únicas
cuisines_per_city = df1.groupby('city')['cuisines'].nunique().reset_index()

# Encontrando a cidade com a maior quantidade de culinárias distintas
most_diverse_city_row = cuisines_per_city.loc[cuisines_per_city['cuisines'].idxmax()]
most_diverse_city_name = most_diverse_city_row['city']
most_diverse_cuisines_count = most_diverse_city_row['cuisines']

print(f"A cidade com a maior quantidade de tipos de culinária distintas é '{most_diverse_city_name}' com {most_diverse_cuisines_count} culinárias.")

A cidade com a maior quantidade de tipos de culinária distintas é 'Birmingham' com 32 culinárias.


In [14]:
# 6. Qual o nome da cidade que possui a maior quantidade de restaurantes que fazem reservas?

# Filtrando restaurantes que aceitam reservas (assumindo 1 para 'Sim' ou True)
restaurants_with_booking_city = df1[df1['has_table_booking'] == 1]

# Agrupando por cidade e contando restaurantes únicos que aceitam reservas
booking_restaurants_per_city = restaurants_with_booking_city.groupby('city')['restaurant_id'].nunique().reset_index()

# Encontrando a cidade com a maior quantidade
if not booking_restaurants_per_city.empty:
    most_booking_city_row = booking_restaurants_per_city.loc[booking_restaurants_per_city['restaurant_id'].idxmax()]
    most_booking_city_name = most_booking_city_row['city']
    most_booking_city_count = most_booking_city_row['restaurant_id']
    print(f"A cidade com a maior quantidade de restaurantes que aceitam reservas é '{most_booking_city_name}' com {most_booking_city_count} restaurantes.")
else:
    print("Não há cidades com restaurantes que aceitam reservas.")

A cidade com a maior quantidade de restaurantes que aceitam reservas é 'Bangalore' com 42 restaurantes.


In [26]:
#7. Qual o nome da cidade que possui a maior quantidade de restaurantes que fazem entregas?

# Filtrando restaurantes que fazem entregas (assumindo 1 para 'Sim' ou True)
restaurants_with_delivery_city = df1[df1['is_delivering_now'] == 1]

# Agrupando por cidade e contando restaurantes únicos que fazem entregas
delivery_restaurants_per_city = restaurants_with_delivery_city.groupby('city')['restaurant_id'].nunique().reset_index()

# Encontrando a cidade com a maior quantidade
if not delivery_restaurants_per_city.empty:
    most_delivery_city_row = delivery_restaurants_per_city.loc[delivery_restaurants_per_city['restaurant_id'].idxmax()]
    most_delivery_city_name = most_delivery_city_row['city']
    most_delivery_city_count = most_delivery_city_row['restaurant_id']
    print(f"A cidade com a maior quantidade de restaurantes que fazem entregas é '{most_delivery_city_name}' com {most_delivery_city_count} restaurantes.")
else:
    print("Não há cidades com restaurantes que fazem entregas.")

A cidade com a maior quantidade de restaurantes que fazem entregas é 'Amritsar' com 48 restaurantes.


In [23]:
#8. Qual o nome da cidade que possui a maior quantidade de restaurantes que aceitam pedidos online?
#Célula para filtrar restaurantes com entrega online:
delivery_restaurants = df1[df1['has_online_delivery'] == 1]
print(f"DataFrame 'delivery_restaurants' created with {len(delivery_restaurants)} restaurants offering online delivery.")
#Célula para encontrar a cidade com mais restaurantes que fazem entrega online:
delivery_restaurants_per_city = delivery_restaurants.groupby('city')['restaurant_id'].nunique().reset_index()
most_delivery_city_row = delivery_restaurants_per_city.loc[delivery_restaurants_per_city['restaurant_id'].idxmax()]
most_delivery_city_name = most_delivery_city_row['city']
most_delivery_count = most_delivery_city_row['restaurant_id']

print(f"A cidade com a maior quantidade de restaurantes que fazem entrega online é '{most_delivery_city_name}' com {most_delivery_count} restaurantes.")

DataFrame 'delivery_restaurants' created with 2428 restaurants offering online delivery.
A cidade com a maior quantidade de restaurantes que fazem entrega online é 'Bhopal' com 75 restaurantes.


# RESTAURANTES

In [14]:
#1. Qual o nome do restaurante que possui a maior quantidade de avaliações?

# Agrupando por restaurante e somando os votos
restaurant_votes = df1.groupby('restaurant_name')['votes'].sum().reset_index()

# Encontrando o restaurante com o maior número de votos
most_voted_restaurant_row = restaurant_votes.loc[restaurant_votes['votes'].idxmax()]
most_voted_restaurant_name = most_voted_restaurant_row['restaurant_name']
most_voted_count = most_voted_restaurant_row['votes']

print(f"O restaurante com a maior quantidade de avaliações é '{most_voted_restaurant_name}' com {most_voted_count} avaliações.")

O restaurante com a maior quantidade de avaliações é 'Domino's Pizza' com 59749 avaliações.


In [15]:
#2. Qual o nome do restaurante com a maior nota média?

# Agrupando por restaurante e calculando a média das avaliações
restaurant_average_rating = df1.groupby('restaurant_name')['aggregate_rating'].mean().reset_index()

# Encontrando o restaurante com a maior nota média
most_rated_restaurant_row = restaurant_average_rating.loc[restaurant_average_rating['aggregate_rating'].idxmax()]
most_rated_restaurant_name = most_rated_restaurant_row['restaurant_name']
most_rated_restaurant_avg_rating = most_rated_restaurant_row['aggregate_rating']

print(f"O restaurante com a maior nota média é '{most_rated_restaurant_name}' com uma média de {most_rated_restaurant_avg_rating:.2f}.")

O restaurante com a maior nota média é '5 Napkin Burger' com uma média de 4.90.


In [19]:
#3. Qual o nome do restaurante que possui o maior valor de uma prato para duas pessoas?

# Agrupando por restaurante e calculando o valor máximo de 'average_cost_for_two'
restaurant_max_cost = df1.groupby('restaurant_name')['average_cost_for_two'].max().reset_index()

# Encontrando o restaurante com o maior valor médio
most_expensive_restaurant_row = restaurant_max_cost.loc[restaurant_max_cost['average_cost_for_two'].idxmax()]
most_expensive_restaurant_name = most_expensive_restaurant_row['restaurant_name']
most_expensive_restaurant_cost = most_expensive_restaurant_row['average_cost_for_two']

print(f"O restaurante com o maior valor de um prato para duas pessoas é '{most_expensive_restaurant_name}' com um custo de {most_expensive_restaurant_cost:.2f}.")

O restaurante com o maior valor de um prato para duas pessoas é 'd'Arry's Verandah Restaurant' com um custo de 25000017.00.


In [20]:
#4. Qual o nome do restaurante de tipo de culinária brasileira que possui a menor média de avaliação?

# Filtrando restaurantes com culinária 'Brazilian'
brazilian_restaurants = df1[df1['cuisines'] == 'Brazilian']

# Agrupando por restaurante e calculando a média das avaliações para culinária brasileira
brazilian_restaurant_average_rating = brazilian_restaurants.groupby('restaurant_name')['aggregate_rating'].mean().reset_index()

# Encontrando o restaurante com a menor nota média
least_rated_brazilian_restaurant_row = brazilian_restaurant_average_rating.loc[brazilian_restaurant_average_rating['aggregate_rating'].idxmin()]
least_rated_brazilian_restaurant_name = least_rated_brazilian_restaurant_row['restaurant_name']
least_rated_brazilian_restaurant_avg_rating = least_rated_brazilian_restaurant_row['aggregate_rating']

print(f"O restaurante de culinária brasileira com a menor nota média é '{least_rated_brazilian_restaurant_name}' com uma média de {least_rated_brazilian_restaurant_avg_rating:.2f}.")

O restaurante de culinária brasileira com a menor nota média é 'Bar do Luiz Fernandes' com uma média de 0.00.


In [23]:
#5. Qual o nome do restaurante de tipo de culinária brasileira, e que é do Brasil, que possui a maior média de avaliação?

# Filtrando restaurantes com culinária 'Brazilian' e no país 'Brazil'
brazilian_restaurants_in_brazil = df1[(df1['cuisines'] == 'Brazilian') & (df1['country_code'] == 'Brazil')]

# Verificando se há restaurantes brasileiros no Brasil
if not brazilian_restaurants_in_brazil.empty:
    # Agrupando por restaurante e calculando a média das avaliações
    brazilian_restaurant_average_rating_brazil = brazilian_restaurants_in_brazil.groupby('restaurant_name')['aggregate_rating'].mean().reset_index()

    # Encontrando o restaurante com a maior nota média
    most_rated_brazilian_restaurant_row_brazil = brazilian_restaurant_average_rating_brazil.loc[brazilian_restaurant_average_rating_brazil['aggregate_rating'].idxmax()]
    most_rated_brazilian_restaurant_name_brazil = most_rated_brazilian_restaurant_row_brazil['restaurant_name']
    most_rated_brazilian_restaurant_avg_rating_brazil = most_rated_brazilian_restaurant_row_brazil['aggregate_rating']

    print(f"O restaurante de culinária brasileira no Brasil com a maior nota média é '{most_rated_brazilian_restaurant_name_brazil}' com uma média de {most_rated_brazilian_restaurant_avg_rating_brazil:.2f}.")
else:
    print("Não há restaurantes de culinária brasileira registrados no Brasil.")

O restaurante de culinária brasileira no Brasil com a maior nota média é 'Aprazível' com uma média de 4.90.


In [25]:
#6. Os restaurantes que aceitam pedido online são também, na média, os restaurantes que mais possuem avaliações registradas?

# Calculando a média de avaliações para restaurantes que aceitam pedido online
average_votes_online_delivery = df1[df1['has_online_delivery'] == 1]['votes'].mean()

# Calculando a média de avaliações para restaurantes que NÃO aceitam pedido online
average_votes_no_online_delivery = df1[df1['has_online_delivery'] == 0]['votes'].mean()

print(f"Média de avaliações para restaurantes com pedido online: {average_votes_online_delivery:.2f}")
print(f"Média de avaliações para restaurantes sem pedido online: {average_votes_no_online_delivery:.2f}")

if average_votes_online_delivery > average_votes_no_online_delivery:
    print("\nConclusão: Sim, os restaurantes que aceitam pedido online possuem, na média, mais avaliações registradas.")
else:
    print("\nConclusão: Não, os restaurantes que aceitam pedido online não possuem, na média, mais avaliações registradas do que os que não aceitam.")

Média de avaliações para restaurantes com pedido online: 838.82
Média de avaliações para restaurantes sem pedido online: 479.63

Conclusão: Sim, os restaurantes que aceitam pedido online possuem, na média, mais avaliações registradas.


In [26]:
#7. Os restaurantes que fazem reservas são também, na média, os restaurantes que possuem o maior valor médio de um prato para duas pessoas?

# Calculando a média do valor de um prato para dois para restaurantes que aceitam reservas
average_cost_with_booking = df1[df1['has_table_booking'] == 1]['average_cost_for_two'].mean()

# Calculando a média do valor de um prato para dois para restaurantes que NÃO aceitam reservas
average_cost_without_booking = df1[df1['has_table_booking'] == 0]['average_cost_for_two'].mean()

print(f"Média do custo para dois em restaurantes com reservas: {average_cost_with_booking:.2f}")
print(f"Média do custo para dois em restaurantes sem reservas: {average_cost_without_booking:.2f}")

if average_cost_with_booking > average_cost_without_booking:
    print("\nConclusão: Sim, os restaurantes que fazem reservas possuem, na média, um maior valor médio para um prato para duas pessoas.")
else:
    print("\nConclusão: Não, os restaurantes que fazem reservas não possuem, na média, um maior valor médio para um prato para duas pessoas do que os que não aceitam.")

Média do custo para dois em restaurantes com reservas: 69998.42
Média do custo para dois em restaurantes sem reservas: 3489.63

Conclusão: Sim, os restaurantes que fazem reservas possuem, na média, um maior valor médio para um prato para duas pessoas.


In [28]:
#8. Os restaurantes do tipo de culinária japonesa dos Estados Unidos da América possuem um valor médio de prato para duas pessoas maior que as churrascarias americanas (BBQ)?

# Filtrando restaurantes japoneses nos EUA
japanese_restaurants_usa = df1[(df1['cuisines'] == 'Japanese') & (df1['country_code'] == 'United States of America')]

# Calculando o valor médio para prato para dois em restaurantes japoneses nos EUA
average_cost_japanese_usa = japanese_restaurants_usa['average_cost_for_two'].mean()

# Filtrando churrascarias americanas (BBQ) nos EUA
bbq_restaurants_usa = df1[(df1['cuisines'] == 'BBQ') & (df1['country_code'] == 'United States of America')]

# Calculando o valor médio para prato para dois em churrascarias americanas (BBQ) nos EUA
average_cost_bbq_usa = bbq_restaurants_usa['average_cost_for_two'].mean()

print(f"Média do custo para dois em restaurantes japoneses nos EUA: {average_cost_japanese_usa:.2f}")
print(f"Média do custo para dois em churrascarias americanas (BBQ) nos EUA: {average_cost_bbq_usa:.2f}")

if average_cost_japanese_usa > average_cost_bbq_usa:
    print("\nConclusão: Sim, os restaurantes de culinária japonesa nos Estados Unidos da América possuem um valor médio de prato para duas pessoas MAIOR que as churrascarias americanas (BBQ).")
elif average_cost_japanese_usa < average_cost_bbq_usa:
    print("\nConclusão: Não, os restaurantes de culinária japonesa nos Estados Unidos da América possuem um valor médio de prato para duas pessoas MENOR que as churrascarias americanas (BBQ).")
else:
    print("\nConclusão: Os restaurantes de culinária japonesa nos Estados Unidos da América e as churrascarias americanas (BBQ) possuem um valor médio de prato para duas pessoas IGUAL.")

Média do custo para dois em restaurantes japoneses nos EUA: 56.41
Média do custo para dois em churrascarias americanas (BBQ) nos EUA: 39.64

Conclusão: Sim, os restaurantes de culinária japonesa nos Estados Unidos da América possuem um valor médio de prato para duas pessoas MAIOR que as churrascarias americanas (BBQ).


# TIPOS DE CULINARIA

In [35]:
def get_max_rated_restaurant_for_cuisine(df, cuisine_type):
    # Filter the DataFrame for the specified cuisine type
    filtered_cuisine = df[df['cuisines'] == cuisine_type]

    # Group by restaurant name and calculate the mean of aggregate_rating
    restaurant_ratings = filtered_cuisine.groupby('restaurant_name')['aggregate_rating'].mean().reset_index()

    # Find the maximum aggregate rating for this cuisine
    if not restaurant_ratings.empty:
        max_rating = restaurant_ratings['aggregate_rating'].max()

        # Filter to select all restaurants with this maximum rating
        top_rated_restaurants = restaurant_ratings[restaurant_ratings['aggregate_rating'] == max_rating]

        return top_rated_restaurants
    else:
        return pd.DataFrame(columns=['restaurant_name', 'aggregate_rating']) # Return empty DataFrame if no restaurants found

print("Function 'get_max_rated_restaurant_for_cuisine' defined successfully.")

Function 'get_max_rated_restaurant_for_cuisine' defined successfully.


In [36]:
def get_min_rated_restaurant_for_cuisine(df, cuisine_type):
    # Filter the DataFrame for the specified cuisine type
    filtered_cuisine = df[df['cuisines'] == cuisine_type]

    # Group by restaurant name and calculate the mean of aggregate_rating
    restaurant_ratings = filtered_cuisine.groupby('restaurant_name')['aggregate_rating'].mean().reset_index()

    # Find the minimum aggregate rating for this cuisine
    if not restaurant_ratings.empty:
        min_rating = restaurant_ratings['aggregate_rating'].min()

        # Filter to select all restaurants with this minimum rating
        bottom_rated_restaurants = restaurant_ratings[restaurant_ratings['aggregate_rating'] == min_rating]

        return bottom_rated_restaurants
    else:
        return pd.DataFrame(columns=['restaurant_name', 'aggregate_rating']) # Return empty DataFrame if no restaurants found

print("Function 'get_min_rated_restaurant_for_cuisine' defined successfully.")

Function 'get_min_rated_restaurant_for_cuisine' defined successfully.


In [45]:
cuisines_to_analyze = ['Italian', 'American', 'Arabian', 'Japanese', 'Brazilian']

max_rated_results = {}
min_rated_results = {}

for cuisine in cuisines_to_analyze:
    max_rated_results[cuisine] = get_max_rated_restaurant_for_cuisine(df1, cuisine)
    min_rated_results[cuisine] = get_min_rated_restaurant_for_cuisine(df1, cuisine)

print("Highest and lowest rated restaurants for specified cuisines have been processed.")

Highest and lowest rated restaurants for specified cuisines have been processed.


In [47]:
print("\n--- Restaurant Ratings Summary by Cuisine ---")

for cuisine in cuisines_to_analyze:
    print(f"\nCuisine: {cuisine}")

    # Display Highest Rated Restaurants
    max_results = max_rated_results[cuisine]
    if not max_results.empty:
        print(f"  Highest Rated Restaurant(s) (Avg Rating: {max_results['aggregate_rating'].iloc[0]:.2f}):")
        for index, row in max_results.iterrows():
            print(f"    - {row['restaurant_name']}")
    else:
        print("  No restaurants found for this cuisine with a rating.")

    # Display Lowest Rated Restaurants
    min_results = min_rated_results[cuisine]
    if not min_results.empty:
        print(f"  Lowest Rated Restaurant(s) (Avg Rating: {min_results['aggregate_rating'].iloc[0]:.2f}):")
        for index, row in min_results.iterrows():
            print(f"    - {row['restaurant_name']}")
    else:
        print("  No restaurants found for this cuisine with a rating.")


--- Restaurant Ratings Summary by Cuisine ---

Cuisine: Italian
  Highest Rated Restaurant(s) (Avg Rating: 4.90):
    - Amano Restaurant
    - Andre's Cucina & Polenta Bar
    - Big Bill's
    - Bottega Louie
    - Cafe Del Sol Classico
    - Celino's
    - Central Grocery
    - Cerroni's Purple Garlic
    - Chicago Pizza & Oven Grinder Company
    - Di Rienzo Grocery & Deli
    - Guillermo's
    - Ombra
    - Osteria Marco
    - Perricone's Marketplace & Café
    - Regina Pizzeria
    - The Parlor Pizzeria
    - Zolocrust - Hotel Clarks Amer
  Lowest Rated Restaurant(s) (Avg Rating: 0.00):
    - Avenida Paulista
    - Bene - Sheraton Rio Hotel
    - La Bocca Bar e Trattoria
    - Le Delicatezze Di Bruno
    - Più
    - Ristorantino
    - The Pasta Factory
    - Veeno

Cuisine: American
  Highest Rated Restaurant(s) (Avg Rating: 4.90):
    - 5 Napkin Burger
    - Brick Store Pub
    - Cafe Muse
    - Cut By Wolfgang Puck
    - Fat Cat
    - Hodad's
    - Kono's Surf Club Cafe
    - La

In [51]:
#11. Qual o tipo de culinária que possui o maior valor médio de um prato para duas pessoas?

# Agrupando por culinária e calculando a média do custo para dois
average_cost_per_cuisine = df1.groupby('cuisines')['average_cost_for_two'].mean().reset_index()

# Encontrando a culinária com o maior valor médio
most_expensive_cuisine_row = average_cost_per_cuisine.loc[average_cost_per_cuisine['average_cost_for_two'].idxmax()]
most_expensive_cuisine_name = most_expensive_cuisine_row['cuisines']
most_expensive_cuisine_avg_cost = most_expensive_cuisine_row['average_cost_for_two']

print(f"O tipo de culinária com o maior valor médio de um prato para duas pessoas é '{most_expensive_cuisine_name}' com um custo médio de {most_expensive_cuisine_avg_cost:.2f}.")

O tipo de culinária com o maior valor médio de um prato para duas pessoas é 'Modern Australian' com um custo médio de 1470693.06.


In [52]:
#12. Qual o tipo de culinária que possui a maior nota média?

# Agrupando por culinária e calculando a média da nota agregada
average_rating_per_cuisine = df1.groupby('cuisines')['aggregate_rating'].mean().reset_index()

# Encontrando a culinária com a maior nota média
most_rated_cuisine_row = average_rating_per_cuisine.loc[average_rating_per_cuisine['aggregate_rating'].idxmax()]
most_rated_cuisine_name = most_rated_cuisine_row['cuisines']
most_rated_cuisine_avg_rating = most_rated_cuisine_row['aggregate_rating']

print(f"O tipo de culinária com a maior nota média é '{most_rated_cuisine_name}' com uma média de {most_rated_cuisine_avg_rating:.2f}.")

O tipo de culinária com a maior nota média é 'Others' com uma média de 4.90.


In [53]:
#13. Qual o tipo de culinária que possui mais restaurantes que aceitam pedidos online e fazem entregas?

# Filtrando restaurantes que aceitam pedidos online e fazem entregas
online_delivery_restaurants = df1[(df1['has_online_delivery'] == 1) & (df1['is_delivering_now'] == 1)]

# Agrupando por culinária e contando restaurantes únicos
cuisine_online_delivery_count = online_delivery_restaurants.groupby('cuisines')['restaurant_id'].nunique().reset_index()

# Encontrando a culinária com o maior número de restaurantes que aceitam pedidos online e fazem entregas
most_online_delivery_cuisine_row = cuisine_online_delivery_count.loc[cuisine_online_delivery_count['restaurant_id'].idxmax()]
most_online_delivery_cuisine_name = most_online_delivery_cuisine_row['cuisines']
most_online_delivery_count = most_online_delivery_cuisine_row['restaurant_id']

print(f"O tipo de culinária com mais restaurantes que aceitam pedidos online e fazem entregas é '{most_online_delivery_cuisine_name}' com {most_online_delivery_count} restaurantes.")

O tipo de culinária com mais restaurantes que aceitam pedidos online e fazem entregas é 'North Indian' com 317 restaurantes.


#Gráficos


In [56]:
#grafico 1 Quantidade de Restaurantes registrados por País
restaurants_per_country_df = df1.groupby('country_code')['restaurant_id'].nunique().reset_index()
restaurants_per_country_df.columns = ['country_code', 'restaurant_count']
restaurants_per_country_df = restaurants_per_country_df.sort_values(by='restaurant_count', ascending=False)

fig = px.bar(
    restaurants_per_country_df,
    x='country_code',
    y='restaurant_count',
    title='Quantidade de Restaurantes Registrados por País',
    labels={'country_code': 'País', 'restaurant_count': 'Número de Restaurantes'},
    text='restaurant_count'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [57]:
#grafico 2 Quantidade de Cidades registradas por País
cities_per_country_df = df1.groupby('country_code')['city'].nunique().reset_index()
cities_per_country_df.columns = ['country_code', 'city_count']
cities_per_country_df = cities_per_country_df.sort_values(by='city_count', ascending=False)

fig = px.bar(
    cities_per_country_df,
    x='country_code',
    y='city_count',
    title='Quantidade de Cidades Registradas por País',
    labels={'country_code': 'País', 'city_count': 'Número de Cidades'},
    text='city_count'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [14]:
#grafico 3 Média da Quantidade de Avaliações Feitas por País
average_votes_per_country_df = df1.groupby('country_code')['votes'].mean().reset_index()
average_votes_per_country_df.columns = ['country_code', 'average_votes']
average_votes_per_country_df = average_votes_per_country_df.sort_values(by='average_votes', ascending=False)

fig = px.bar(
    average_votes_per_country_df,
    x='country_code',
    y='average_votes',
    title='Média da Quantidade de Avaliações Feitas por País',
    labels={'country_code': 'País', 'average_votes': 'Média de Votos'},
    text='average_votes'
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)


In [15]:
#grafico 4 Média de preço de um prato para duas pessoas por País
average_cost_for_two_per_country_df = df1.groupby('country_code')['average_cost_for_two'].mean().reset_index()
average_cost_for_two_per_country_df.columns = ['country_code', 'average_cost_for_two']
average_cost_for_two_per_country_df = average_cost_for_two_per_country_df.sort_values(by='average_cost_for_two', ascending=False)

fig = px.bar(
    average_cost_for_two_per_country_df,
    x='country_code',
    y='average_cost_for_two',
    title='Média de Preço de um Prato para Duas Pessoas por País',
    labels={'country_code': 'País', 'average_cost_for_two': 'Média de Custo'},
    text='average_cost_for_two'
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [17]:
#grafico 1 Top 10 Cidades com mais restaurantes
restaurants_per_city_df = df1.groupby('city')['restaurant_id'].nunique().reset_index()
restaurants_per_city_df.columns = ['city', 'restaurant_count']
restaurants_per_city_df = restaurants_per_city_df.sort_values(by='restaurant_count', ascending=False).head(10)

fig = px.bar(
    restaurants_per_city_df,
    x='city',
    y='restaurant_count',
    title='Top 10 Cidades com Mais Restaurantes',
    labels={'city': 'Cidade', 'restaurant_count': 'Número de Restaurantes'},
    text='restaurant_count'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)


In [18]:
#grafico 2 Top 7 Cidades com mais restaurantes com média de avaliação acima de 4
restaurants_above_4_rating = df1[df1['aggregate_rating'] > 4]
restaurants_above_4_per_city = restaurants_above_4_rating.groupby('city')['restaurant_id'].nunique().reset_index()
restaurants_above_4_per_city.columns = ['city', 'restaurant_count']
restaurants_above_4_per_city = restaurants_above_4_per_city.sort_values(by='restaurant_count', ascending=False).head(7)

fig = px.bar(
    restaurants_above_4_per_city,
    x='city',
    y='restaurant_count',
    title='Top 7 Cidades com Mais Restaurantes (Média de Avaliação Acima de 4)',
    labels={'city': 'Cidade', 'restaurant_count': 'Número de Restaurantes'},
    text='restaurant_count'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)


In [19]:
#grafico 3 Top 7 Cidades com mais restauresntes com média de avaliação abaixo de 2.5
restaurants_below_2_5_rating = df1[df1['aggregate_rating'] < 2.5]
restaurants_below_2_5_per_city = restaurants_below_2_5_rating.groupby('city')['restaurant_id'].nunique().reset_index()
restaurants_below_2_5_per_city.columns = ['city', 'restaurant_count']
restaurants_below_2_5_per_city = restaurants_below_2_5_per_city.sort_values(by='restaurant_count', ascending=False).head(7)

fig = px.bar(
    restaurants_below_2_5_per_city,
    x='city',
    y='restaurant_count',
    title='Top 7 Cidades com Mais Restaurantes (Média de Avaliação Abaixo de 2.5)',
    labels={'city': 'Cidade', 'restaurant_count': 'Número de Restaurantes'},
    text='restaurant_count'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)


In [20]:
#grafico 4 Top 10 Cidades com mais restaurantes com tipos de culinárias distintos
cuisines_per_city_df = df1.groupby('city')['cuisines'].nunique().reset_index()
cuisines_per_city_df.columns = ['city', 'cuisine_count']
cuisines_per_city_df = cuisines_per_city_df.sort_values(by='cuisine_count', ascending=False).head(10)

fig = px.bar(
    cuisines_per_city_df,
    x='city',
    y='cuisine_count',
    title='Top 10 Cidades com Mais Tipos de Culinárias Distintos',
    labels={'city': 'Cidade', 'cuisine_count': 'Número de Tipos de Culinária'},
    text='cuisine_count'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)


In [18]:
# Gráfico Top 10 Melhores Tipos de Culinárias por Média de Avaliação

# Agrupando por culinária e calculando a média da nota agregada
average_rating_per_cuisine = df1.groupby('cuisines')['aggregate_rating'].mean().reset_index()

# Ordenando e selecionando os top 10
top_10_cuisines = average_rating_per_cuisine.sort_values(by='aggregate_rating', ascending=False).head(10)

fig = px.bar(
    top_10_cuisines,
    x='cuisines',
    y='aggregate_rating',
    title='Top 10 Melhores Tipos de Culinárias (Média de Avaliação)',
    labels={'cuisines': 'Tipo de Culinária', 'aggregate_rating': 'Média de Avaliação'},
    text='aggregate_rating'
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)

In [ ]:
# Gráfico Top 10 Piores Tipos de Culinárias por Média de Avaliação

# Agrupando por culinária e calculando a média da nota agregada
average_rating_per_cuisine = df1.groupby('cuisines')['aggregate_rating'].mean().reset_index()

# Ordenando e selecionando os top 10 piores (menores notas)
top_10_worst_cuisines = average_rating_per_cuisine.sort_values(by='aggregate_rating', ascending=True).head(10)

fig = px.bar(
    top_10_worst_cuisines,
    x='cuisines',
    y='aggregate_rating',
    title='Top 10 Piores Tipos de Culinárias (Média de Avaliação)',
    labels={'cuisines': 'Tipo de Culinária', 'aggregate_rating': 'Média de Avaliação'},
    text='aggregate_rating'
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45)
fig.show()